In [ ]:
# Verified Experiments Analysis
# Try to load verified experiment results
from pathlib import Path

verified_results_path = Path('../experiment_results')  # Default output directory

print("📊 Verified Experiment Analysis")
print("="*50)

# Check for verified experiment result files
if verified_results_path.exists():
    result_files = list(verified_results_path.glob('verified_*.json'))
    if result_files:
        print(f"Found {len(result_files)} verified experiment result file(s)")
        
        # Load and display results
        verified_data = []
        for f in result_files[-5:]:  # Show last 5
            try:
                with open(f) as fp:
                    data = json.load(fp)
                    verified_data.append({
                        'file': f.name,
                        'model': data.get('config', {}).get('model_id', 'unknown'),
                        'iterations': data.get('config', {}).get('iterations', 0),
                        'accuracy': data.get('summary', {}).get('accuracy', 0),
                        'total_cost': data.get('summary', {}).get('total_cost', 0)
                    })
            except Exception as e:
                pass
        
        if verified_data:
            verified_df = pd.DataFrame(verified_data)
            display(verified_df)
            
            # Plot accuracy comparison if multiple models
            if len(verified_df['model'].unique()) > 1:
                fig = px.bar(
                    verified_df,
                    x='model',
                    y='accuracy',
                    title='Verified Experiment Accuracy by Model',
                    labels={'accuracy': 'Accuracy', 'model': 'Model'},
                    color='model',
                    color_discrete_map={'gemini-2.5-flash': '#4ecdc4', 'gemini-2.5-pro': '#ff6b6b'}
                )
                fig.update_layout(yaxis_range=[0, 1])
                fig.show()
    else:
        print("⚠️ No verified experiment results found.")
        print("To run verified experiments:")
        print("  python3 -m src.experiments.verified_experiment --compare-models -n 20")
        print("  python3 -m src.experiments.verified_experiment --compare-models -d hard -n 10")
else:
    print("⚠️ Experiment results directory not found.")
    print("To run verified experiments:")
    print("  python3 -m src.experiments.verified_experiment --compare-models -n 20")

## 14. Verified Experiments (Ground Truth Accuracy)

Analyze experiments with verifiable correct answers to measure true model accuracy.

In [ ]:
# Domain-Specific Analysis
# Check for domain experiment data (stored with 'domain' in workflow or pipeline name)
domain_runs = runs_df[
    (runs_df['workflow'].str.contains('domain', case=False, na=False)) |
    (runs_df['pipeline'].str.contains('domain', case=False, na=False))
]

if domain_runs.empty:
    print("⚠️ No domain experiment data available.")
    print("To run domain experiments:")
    print("  python3 -m src.experiments.domain_experiment --compare-domains")
    print("\nAvailable domains: coding, biology, legal, creative, finance, medical, general, complex_reasoning")
else:
    print("📊 Domain Experiment Analysis")
    print("="*50)
    
    # If domain info is available in a column
    if 'domain' in domain_runs.columns:
        domain_summary = domain_runs.groupby(['domain', 'model']).agg({
            'total_cost': ['mean', 'std'],
            'combined_score': 'mean' if 'combined_score' in domain_runs.columns else 'count'
        }).round(4)
        display(domain_summary)
        
        fig = px.bar(
            domain_summary.reset_index(),
            x='domain',
            y=('total_cost', 'mean'),
            color='model',
            barmode='group',
            title='Cost by Domain and Model',
            color_discrete_map={'gemini-2.5-flash': '#4ecdc4', 'gemini-2.5-pro': '#ff6b6b'}
        )
        fig.show()
    else:
        print(f"Found {len(domain_runs)} domain-related runs")
        display(domain_runs.groupby(['pipeline', 'model'])['total_cost'].describe())

## 13. Domain-Specific Analysis

Analyze performance across specialized domains (coding, biology, legal, etc.).

In [ ]:
# Pareto Frontier Analysis
def find_pareto_optimal(df, cost_col='avg_cost', quality_col='avg_quality'):
    """Find Pareto-optimal configurations (lower cost, higher quality)."""
    pareto_mask = np.ones(len(df), dtype=bool)
    for i, (cost_i, qual_i) in enumerate(zip(df[cost_col], df[quality_col])):
        for j, (cost_j, qual_j) in enumerate(zip(df[cost_col], df[quality_col])):
            if i != j:
                # j dominates i if j has lower/equal cost AND higher/equal quality (with at least one strict)
                if cost_j <= cost_i and qual_j >= qual_i and (cost_j < cost_i or qual_j > qual_i):
                    pareto_mask[i] = False
                    break
    return pareto_mask

# Check if we have quality data
if not quality_df.empty and not runs_df.empty:
    # Merge runs with quality
    runs_quality = runs_df.merge(quality_df, left_on='id', right_on='run_id', how='inner')
    
    if 'combined_score' in runs_quality.columns and runs_quality['combined_score'].notna().any():
        # Aggregate by pipeline and model
        pareto_data = runs_quality.groupby(['pipeline', 'model']).agg({
            'total_cost': 'mean',
            'combined_score': 'mean'
        }).reset_index()
        pareto_data.columns = ['pipeline', 'model', 'avg_cost', 'avg_quality']
        
        # Find Pareto-optimal points
        pareto_data['is_pareto'] = find_pareto_optimal(pareto_data)
        
        print("📊 Pareto Frontier Analysis")
        print("="*50)
        print(f"Total configurations: {len(pareto_data)}")
        print(f"Pareto-optimal configurations: {pareto_data['is_pareto'].sum()}")
        
        # Show Pareto-optimal configurations
        pareto_optimal = pareto_data[pareto_data['is_pareto']].sort_values('avg_cost')
        print("\n★ Pareto-Optimal Configurations (no configuration dominates these):")
        display(pareto_optimal[['pipeline', 'model', 'avg_cost', 'avg_quality']])
        
        # Scatter plot with Pareto frontier
        fig = px.scatter(
            pareto_data,
            x='avg_cost',
            y='avg_quality',
            color='model',
            symbol='is_pareto',
            symbol_map={True: 'star', False: 'circle'},
            hover_data=['pipeline'],
            title='Cost vs Quality with Pareto Frontier',
            labels={'avg_cost': 'Average Cost ($)', 'avg_quality': 'Average Quality Score'},
            color_discrete_map={'gemini-2.5-flash': '#4ecdc4', 'gemini-2.5-pro': '#ff6b6b'}
        )
        
        # Add Pareto frontier line
        pareto_line = pareto_optimal.sort_values('avg_cost')
        fig.add_trace(go.Scatter(
            x=pareto_line['avg_cost'],
            y=pareto_line['avg_quality'],
            mode='lines',
            name='Pareto Frontier',
            line=dict(color='gold', width=2, dash='dash')
        ))
        
        fig.update_traces(marker=dict(size=12), selector=dict(mode='markers'))
        fig.show()
    else:
        print("⚠️ No quality scores available for Pareto analysis.")
        print("Run experiments with --llm-eval flag to enable quality scoring.")
else:
    print("⚠️ Quality data not available for Pareto analysis.")

## 12. Pareto Frontier Analysis

Identify optimal cost-quality configurations using Pareto frontier.

In [ ]:
# Input/Output Token Ratios by Stage Type
if not stages_df.empty:
    # Calculate output/input ratio for each stage type
    stage_ratios = stages_df.groupby('stage_type').apply(
        lambda x: pd.Series({
            'output_input_ratio': x['output_tokens'].sum() / max(1, x['input_tokens'].sum()),
            'avg_input': x['input_tokens'].mean(),
            'avg_output': x['output_tokens'].mean(),
            'count': len(x)
        })
    ).round(2)
    stage_ratios = stage_ratios.sort_values('output_input_ratio', ascending=False)
    
    print("📊 Token Ratio by Stage Type (Output/Input)")
    print("Higher ratio = more output amplification")
    display(stage_ratios)
    
    # Bar chart of ratios
    fig = px.bar(
        stage_ratios.reset_index(),
        x='stage_type',
        y='output_input_ratio',
        title='Output/Input Token Ratio by Stage Type',
        labels={'output_input_ratio': 'Output/Input Ratio', 'stage_type': 'Stage Type'},
        color='output_input_ratio',
        color_continuous_scale='Teal'
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

In [ ]:
# Token Distribution Analysis
if not stages_df.empty:
    print("📊 Token Distribution by Workflow")
    print("="*50)
    
    # Join with runs to get workflow info
    stages_with_workflow = stages_df.merge(
        runs_df[['id', 'workflow', 'model']], 
        left_on='run_id', 
        right_on='id', 
        how='left'
    )
    
    # Token stats by workflow
    token_stats = stages_with_workflow.groupby('workflow').agg({
        'input_tokens': ['mean', 'median', 'std', 'max'],
        'output_tokens': ['mean', 'median', 'std', 'max']
    }).round(0)
    token_stats.columns = ['input_mean', 'input_median', 'input_std', 'input_max',
                          'output_mean', 'output_median', 'output_std', 'output_max']
    display(token_stats)
    
    # Token distribution histogram
    fig = make_subplots(rows=1, cols=2, subplot_titles=('Input Tokens', 'Output Tokens'))
    
    for i, model in enumerate(['gemini-2.5-flash', 'gemini-2.5-pro']):
        model_data = stages_with_workflow[stages_with_workflow['model'] == model]
        color = '#4ecdc4' if model == 'gemini-2.5-flash' else '#ff6b6b'
        
        fig.add_trace(
            go.Histogram(x=model_data['input_tokens'], name=f'{model} input', 
                        marker_color=color, opacity=0.7),
            row=1, col=1
        )
        fig.add_trace(
            go.Histogram(x=model_data['output_tokens'], name=f'{model} output',
                        marker_color=color, opacity=0.7),
            row=1, col=2
        )
    
    fig.update_layout(title='Token Distribution by Model', barmode='overlay')
    fig.show()
else:
    print("⚠️ No stage data available for token profiling.")

## 11. Token Profiler

Analyze token distribution patterns across workflows and stages.

In [ ]:
# RAG Stage Cost Breakdown
if not rag_runs.empty:
    # Get stage-level data for RAG runs
    rag_run_ids = rag_runs['id'].tolist()
    rag_stages = stages_df[stages_df['run_id'].isin(rag_run_ids)]
    
    if not rag_stages.empty:
        stage_breakdown = rag_stages.groupby('stage_type').agg({
            'cost': ['sum', 'mean'],
            'input_tokens': 'mean',
            'output_tokens': 'mean'
        }).round(4)
        stage_breakdown.columns = ['total_cost', 'avg_cost', 'avg_input', 'avg_output']
        stage_breakdown = stage_breakdown.sort_values('total_cost', ascending=False)
        
        print("RAG Stage Cost Breakdown:")
        display(stage_breakdown)
        
        # Pie chart of stage costs
        fig = px.pie(
            values=stage_breakdown['total_cost'].values,
            names=stage_breakdown.index,
            title='RAG Cost Distribution by Stage',
            color_discrete_sequence=px.colors.qualitative.Set2
        )
        fig.show()

In [ ]:
# RAG Pipeline Analysis
rag_runs = runs_df[runs_df['workflow'] == 'rag']

if rag_runs.empty:
    print("⚠️ No RAG experiment data available.")
    print("To run RAG experiments:")
    print("  1. Generate corpus: python3 scripts/generate_academic_corpus.py --chunks 200")
    print("  2. Build index: python3 scripts/build_rag_index.py")
    print("  3. Run experiment: python3 -m src.experiment --workflow rag --iterations 5")
else:
    print(f"📊 RAG Experiment Overview")
    print(f"{'='*40}")
    print(f"Total RAG runs: {len(rag_runs):,}")
    print(f"Total RAG cost: {format_cost(rag_runs['total_cost'].sum())}")
    
    # RAG variant comparison
    rag_summary = rag_runs.groupby(['pipeline', 'model']).agg({
        'total_cost': ['mean', 'std', 'count'],
        'total_latency_ms': 'mean',
        'input_tokens': 'sum',
        'output_tokens': 'sum'
    }).round(6)
    rag_summary.columns = ['avg_cost', 'std_cost', 'runs', 'avg_latency', 'total_input', 'total_output']
    rag_summary = rag_summary.reset_index()
    
    print("\nRAG Pipeline Comparison:")
    display(rag_summary)
    
    # Visualize RAG costs
    fig = px.bar(
        rag_summary,
        x='pipeline',
        y='avg_cost',
        color='model',
        barmode='group',
        error_y='std_cost',
        title='RAG Pipeline Cost Comparison',
        labels={'avg_cost': 'Average Cost ($)', 'pipeline': 'Pipeline Variant'},
        color_discrete_map={'gemini-2.5-flash': '#4ecdc4', 'gemini-2.5-pro': '#ff6b6b'}
    )
    fig.show()

# LLM Cost Decomposition Analysis

Comprehensive analysis of multi-stage pipelines, agentic workflows, A/B testing, and security document analysis.

## Analyses:
1. **Pipeline Cost Comparison** - Compare costs across pipeline types
2. **Stage-Level Attribution** - Where does the money go?
3. **Model Comparison** - Flash vs Pro economics
4. **Agentic Patterns** - ReAct iterations, multi-turn growth
5. **Streaming Metrics** - TTFT and throughput analysis
6. **A/B Testing** - Prompt variant performance
7. **Document Analysis** - Vulnerability detection accuracy
8. **Cost-Quality Tradeoffs** - Value optimization
9. **RAG Pipeline Analysis** - Retrieval-augmented generation costs
10. **Token Profiler** - Token distribution patterns
11. **Pareto Frontier** - Optimal cost-quality configurations
12. **Domain Analysis** - Cross-domain performance (if available)
13. **Verified Experiments** - Ground truth accuracy (if available)

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import json

import sys
sys.path.insert(0, '..')

# Database queries
from src.db import (
    get_runs, 
    get_stages, 
    get_quality_scores,
    get_pipeline_summary,
    get_stage_summary,
    get_cost_by_stage_type,
    get_cost_by_model,
    get_iteration_analysis,
    get_context_growth_analysis,
    get_streaming_analysis,
    # A/B test queries
    get_ab_test_summary,
    get_ab_test_quality,
    get_ab_test_cost_quality_ratio,
    get_ab_tests,
)

# Vulnerability ground truth
from src.evaluation import (
    get_vulnerabilities,
    get_all_document_ids,
    count_by_severity,
    calculate_detection_score,
)

from src.utils import format_cost

# Plotly theme
import plotly.io as pio
pio.templates.default = 'plotly_white'

## 1. Load Data

In [2]:
runs_df = get_runs(success_only=True)
stages_df = get_stages()
quality_df = get_quality_scores()

print(f"📊 Dataset Overview")
print(f"{'='*40}")
print(f"Total pipeline runs: {len(runs_df):,}")
print(f"Total stage executions: {len(stages_df):,}")
print(f"Quality evaluations: {len(quality_df):,}")

if len(runs_df) > 0:
    print(f"\nPipeline types: {runs_df['pipeline_type'].unique().tolist()}")
    print(f"Workflows: {runs_df['workflow'].unique().tolist()}")
    print(f"Models: {runs_df['model'].unique().tolist()}")
    print(f"\nTotal cost: {format_cost(runs_df['total_cost'].sum())}")

📊 Dataset Overview
Total pipeline runs: 736
Total stage executions: 1,753
Quality evaluations: 735

Pipeline types: ['linear', 'react', 'multiturn', 'self_correcting', 'ab_test']
Workflows: ['verbosity', 'context', 'react', 'multiturn', 'self_correcting', 'document', 'ab_test']
Models: ['gemini-2.5-flash', 'gemini-2.5-pro']

Total cost: $8.07


In [3]:
# Pipeline summary
pipeline_summary = get_pipeline_summary()
if not pipeline_summary.empty:
    print("Pipeline Summary:")
    display(pipeline_summary)

Pipeline Summary:


,pipeline,pipeline_type,model,runs,avg_cost,avg_latency_ms,avg_input_tokens,avg_output_tokens,avg_iterations
0,ab_generation,ab_test,gemini-2.5-flash,80,0.000909,17648.450000,23.300000,1508.512500,1.00
1,ab_generation,ab_test,gemini-2.5-pro,56,0.006824,30958.446429,23.857143,1358.928571,1.00
2,context_long,linear,gemini-2.5-flash,20,0.002015,41828.250000,2707.400000,2682.150000,1.00
3,context_long,linear,gemini-2.5-pro,20,0.014181,70124.450000,2471.900000,2218.250000,1.00
4,context_short,linear,gemini-2.5-flash,20,0.000150,9341.500000,261.250000,184.400000,1.00
5,context_short,linear,gemini-2.5-pro,20,0.001126,21119.350000,252.450000,162.150000,1.00
6,doc_analysis_hybrid,linear,gemini-2.5-flash,20,0.017726,95427.500000,3826.500000,8267.050000,1.00
7,doc_analysis_hybrid,linear,gemini-2.5-pro,20,0.017599,95151.250000,3931.050000,8134.850000,1.00
8,doc_analysis_iterative,linear,gemini-2.5-flash,20,0.014313,97717.700000,4324.650000,6938.200000,1.00
9,doc_analysis_iterative,linear,gemini-2.5-pro,20,0.029987,123374.750000,3506.400000,5120.850000,1.00


## 2. Pipeline Cost Comparison

In [4]:
if not runs_df.empty:
    # Cost by pipeline and model
    cost_comparison = runs_df.groupby(['pipeline', 'model']).agg({
        'total_cost': ['mean', 'std', 'count'],
        'total_latency_ms': 'mean'
    }).round(6)
    cost_comparison.columns = ['avg_cost', 'std_cost', 'runs', 'avg_latency_ms']
    cost_comparison = cost_comparison.reset_index()
    
    fig = px.bar(
        cost_comparison,
        x='pipeline',
        y='avg_cost',
        color='model',
        barmode='group',
        error_y='std_cost',
        title='Average Cost by Pipeline and Model',
        labels={'avg_cost': 'Average Cost ($)', 'pipeline': 'Pipeline'},
        color_discrete_map={'gemini-2.5-flash': '#4ecdc4', 'gemini-2.5-pro': '#ff6b6b'}
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

In [5]:
if not runs_df.empty:
    # Cost distribution boxplot
    fig = px.box(
        runs_df,
        x='pipeline',
        y='total_cost',
        color='model',
        title='Cost Distribution by Pipeline',
        labels={'total_cost': 'Total Cost ($)', 'pipeline': 'Pipeline'}
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

## 3. Stage-Level Cost Attribution

In [6]:
# Cost by stage type
stage_costs = get_cost_by_stage_type()
if not stage_costs.empty:
    fig = px.sunburst(
        stage_costs,
        path=['model', 'stage_type'],
        values='total_cost',
        title='Cost Distribution by Stage Type and Model'
    )
    fig.show()
    
    display(stage_costs)

,stage_type,model,total_cost,avg_cost,count
0,conversation,gemini-2.5-pro,2.013461,0.012584,160
1,generation,gemini-2.5-pro,1.654240,0.008440,196
2,critique,gemini-2.5-pro,0.901604,0.009016,100
3,refinement,gemini-2.5-pro,0.837070,0.013951,60
4,evaluation,gemini-2.5-pro,0.478956,0.007983,60
5,extraction,gemini-2.5-pro,0.445604,0.005570,80
6,thinking,gemini-2.5-pro,0.419532,0.005448,77
7,refinement,gemini-2.5-flash,0.354530,0.002348,151
8,generation,gemini-2.5-flash,0.304209,0.001014,300
9,conversation,gemini-2.5-flash,0.229074,0.001432,160


In [7]:
if not stages_df.empty:
    # Stage type breakdown
    stage_summary = stages_df.groupby('stage_type').agg({
        'cost': ['sum', 'mean', 'count'],
        'input_tokens': 'mean',
        'output_tokens': 'mean'
    }).round(4)
    stage_summary.columns = ['total_cost', 'avg_cost', 'count', 'avg_input_tokens', 'avg_output_tokens']
    stage_summary = stage_summary.sort_values('total_cost', ascending=False)
    display(stage_summary)

,total_cost,avg_cost,count,avg_input_tokens,avg_output_tokens
stage_type,,,,,
conversation,2.2425,0.0070,320,4451.6812,1338.5969
generation,1.9584,0.0039,496,390.8145,1591.5302
refinement,1.1916,0.0056,211,1599.7630,3193.8957
critique,0.9190,0.0077,120,1997.2417,1244.8083
extraction,0.5278,0.0026,200,479.8000,1010.7950
evaluation,0.4940,0.0062,80,1416.3500,1156.9250
thinking,0.4387,0.0037,117,623.1880,834.1966
validation,0.1976,0.0022,91,1556.8132,443.1648
summarization,0.0920,0.0011,80,523.8875,315.0875


## 4. Model Economics: Flash vs Pro

In [8]:
model_costs = get_cost_by_model()
if not model_costs.empty:
    display(model_costs)
    
    # Cost per 1K tokens comparison
    model_costs['cost_per_1k_input'] = model_costs['total_cost'] / (model_costs['total_input_tokens'] / 1000)
    model_costs['cost_per_1k_output'] = model_costs['total_cost'] / (model_costs['total_output_tokens'] / 1000)
    
    fig = px.bar(
        model_costs,
        x='model',
        y=['total_cost'],
        title='Total Cost by Model',
        labels={'value': 'Cost ($)', 'model': 'Model'}
    )
    fig.show()

,model,runs,total_cost,avg_cost,total_input_tokens,total_output_tokens
0,gemini-2.5-flash,380,2.053215,0.005403,1345959,1353002
1,gemini-2.5-pro,356,6.021760,0.016915,1339678,1162005


## 5. Agentic Workflow Analysis

In [9]:
# ReAct iteration analysis
iteration_data = get_iteration_analysis()
if not iteration_data.empty:
    print("ReAct & Self-Correcting Iteration Analysis:")
    display(iteration_data)
    
    # Termination reason distribution
    fig = px.pie(
        iteration_data,
        values='runs',
        names='termination_reason',
        title='Termination Reasons Distribution'
    )
    fig.show()

ReAct & Self-Correcting Iteration Analysis:


,pipeline,model,runs,avg_iterations,min_iterations,max_iterations,avg_cost,cost_per_iteration,termination_reason,termination_pct
0,react_hybrid,gemini-2.5-flash,19,1.894737,1,5,0.010043,0.005364,confidence_reached,95.0
1,react_hybrid,gemini-2.5-flash,1,5.000000,5,5,0.036300,0.007260,max_iterations,5.0
2,react_hybrid,gemini-2.5-pro,20,1.800000,1,4,0.010286,0.005593,confidence_reached,100.0
3,react_research,gemini-2.5-flash,20,1.000000,1,1,0.000467,0.000467,confidence_reached,100.0
4,react_research,gemini-2.5-pro,20,1.000000,1,1,0.000491,0.000491,confidence_reached,100.0
5,self_correcting,gemini-2.5-flash,20,1.150000,1,3,0.001791,0.001505,validation_passed,100.0
6,self_correcting,gemini-2.5-pro,20,1.100000,1,2,0.001602,0.001438,validation_passed,100.0
7,self_correcting_hybrid,gemini-2.5-flash,20,1.150000,1,2,0.006068,0.005034,validation_passed,100.0
8,self_correcting_hybrid,gemini-2.5-pro,20,1.150000,1,2,0.004661,0.003994,validation_passed,100.0


In [10]:
# Multi-turn context growth
context_growth = get_context_growth_analysis()
if not context_growth.empty:
    print("Multi-Turn Context Growth:")
    display(context_growth)
    
    fig = px.line(
        context_growth,
        x='turn',
        y='avg_input_tokens',
        color='model',
        markers=True,
        title='Context Token Growth Across Turns',
        labels={'turn': 'Turn Number', 'avg_input_tokens': 'Average Input Tokens'}
    )
    fig.show()
    
    # Cost growth
    fig2 = px.line(
        context_growth,
        x='turn',
        y='avg_cost',
        color='model',
        markers=True,
        title='Cost Growth Across Turns',
        labels={'turn': 'Turn Number', 'avg_cost': 'Average Cost ($)'}
    )
    fig2.show()

Multi-Turn Context Growth:


,pipeline,model,turn,avg_input_tokens,avg_cost
0,multiturn_3,gemini-2.5-flash,1,6.40,0.000761
1,multiturn_3,gemini-2.5-flash,2,2478.20,0.001799
2,multiturn_3,gemini-2.5-flash,3,7172.75,0.001866
3,multiturn_3,gemini-2.5-pro,1,6.40,0.007273
4,multiturn_3,gemini-2.5-pro,2,2850.30,0.012737
5,multiturn_3,gemini-2.5-pro,3,6458.95,0.014743
6,multiturn_5,gemini-2.5-flash,1,6.40,0.000763
7,multiturn_5,gemini-2.5-flash,2,2488.80,0.001042
8,multiturn_5,gemini-2.5-flash,3,4664.60,0.001561
9,multiturn_5,gemini-2.5-flash,4,7477.30,0.001861


## 6. Streaming Metrics (TTFT)

In [11]:
streaming_data = get_streaming_analysis()
if not streaming_data.empty:
    print("Streaming Metrics:")
    display(streaming_data)
    
    fig = px.bar(
        streaming_data,
        x='pipeline',
        y='avg_ttft_ms',
        color='model',
        barmode='group',
        title='Time to First Token (TTFT) by Pipeline',
        labels={'avg_ttft_ms': 'Avg TTFT (ms)', 'pipeline': 'Pipeline'}
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()
else:
    print("No streaming data available. Run experiments with --streaming flag.")

Streaming Metrics:


,pipeline,model,stage_type,avg_ttft_ms,avg_total_latency_ms,avg_throughput,samples
0,context_long,gemini-2.5-flash,evaluation,13181.75,17621.05,62.4090,20
1,context_long,gemini-2.5-flash,extraction,8576.65,12233.65,80.0225,20
2,context_long,gemini-2.5-flash,summarization,9480.00,11973.55,52.5685,20
3,context_long,gemini-2.5-pro,evaluation,20554.00,27674.35,31.1395,20
4,context_long,gemini-2.5-pro,extraction,14013.00,20714.00,42.8815,20
...,...,...,...,...,...,...,...
65,verbosity_cot,gemini-2.5-flash,generation,10368.15,22337.05,95.2530,20
66,verbosity_cot,gemini-2.5-flash,refinement,7688.85,20416.65,106.2050,20
67,verbosity_cot,gemini-2.5-pro,critique,22261.75,33734.55,34.7965,20
68,verbosity_cot,gemini-2.5-pro,generation,20563.90,39282.75,46.3520,20


## 7. A/B Testing Analysis

Compare different prompt variants to find optimal prompting strategies.

In [ ]:
## 15. Key Findings & Recommendations

In [ ]:
print("\n" + "="*60)
print("📋 KEY FINDINGS SUMMARY")
print("="*60)

if not runs_df.empty:
    # Most expensive pipeline
    pipeline_costs = runs_df.groupby('pipeline')['total_cost'].mean().sort_values(ascending=False)
    print(f"\n💸 Most expensive pipeline: {pipeline_costs.index[0]} ({format_cost(pipeline_costs.iloc[0])}/run)")
    print(f"💵 Cheapest pipeline: {pipeline_costs.index[-1]} ({format_cost(pipeline_costs.iloc[-1])}/run)")
    
    # Model comparison
    model_avg = runs_df.groupby('model')['total_cost'].mean()
    if len(model_avg) >= 2:
        ratio = model_avg.max() / model_avg.min()
        print(f"\n🤖 Model cost ratio: {ratio:.1f}x ({model_avg.idxmax()} vs {model_avg.idxmin()})")
    
    # Total spent
    total = runs_df['total_cost'].sum()
    print(f"\n💰 Total experiment cost: {format_cost(total)}")
    print(f"📊 Total runs: {len(runs_df):,}")
    print(f"📈 Average cost per run: {format_cost(total/len(runs_df))}")
    
    # Workflow coverage
    workflows = runs_df['workflow'].unique().tolist()
    print(f"\n🔬 Workflows analyzed: {', '.join(workflows)}")

# A/B Test insights
if 'ab_summary' in dir() and not ab_summary.empty:
    print("\n🧪 A/B Test Insights:")
    best_variant = ab_summary.loc[ab_summary['avg_cost'].idxmin()]
    print(f"   Most cost-efficient variant: {best_variant['prompt_variant']}")

# Pareto insights
if 'pareto_optimal' in dir() and not pareto_optimal.empty:
    print("\n⭐ Pareto-Optimal Configurations:")
    for _, row in pareto_optimal.head(3).iterrows():
        print(f"   {row['pipeline']} ({row['model']}): ${row['avg_cost']:.4f}, quality={row['avg_quality']:.1f}")

# RAG status
if 'rag_runs' in dir():
    if rag_runs.empty:
        print("\n📚 RAG: No data (run --workflow rag after building index)")
    else:
        print(f"\n📚 RAG: {len(rag_runs)} runs analyzed")

print("\n" + "="*60)

In [14]:
# A/B test quality comparison
ab_quality = get_ab_test_quality()
if not ab_quality.empty:
    print("\n📈 A/B Test Quality Summary:")
    display(ab_quality)
    
    fig = px.bar(
        ab_quality,
        x='prompt_variant',
        y='avg_combined_score',
        color='model',
        facet_col='ab_test_name',
        title='Quality Score by Prompt Variant',
        labels={'avg_combined_score': 'Average Quality Score', 'prompt_variant': 'Variant'}
    )
    fig.show()


📈 A/B Test Quality Summary:


,ab_test_name,prompt_variant,model,runs,avg_automated_score,avg_llm_score,avg_combined_score,avg_response_length,avg_vocabulary_richness
0,generation_style,concise,gemini-2.5-pro,20,96.000000,9.266500,93.999000,584.000000,0.724220
1,generation_style,concise,gemini-2.5-flash,20,87.500000,9.250500,90.503000,461.050000,0.810197
2,generation_style,control,gemini-2.5-flash,20,71.322500,9.850500,87.631500,6716.900000,0.427474
3,generation_style,cot,gemini-2.5-flash,20,68.728000,9.934000,87.094000,8542.750000,0.384551
4,generation_style,detailed,gemini-2.5-pro,14,68.045000,9.976429,87.077143,10295.714286,0.360868
5,generation_style,cot,gemini-2.5-pro,13,68.390769,9.923846,86.898462,8093.615385,0.367818
6,generation_style,control,gemini-2.5-pro,9,68.320000,9.926667,86.887778,7227.777778,0.388628
7,generation_style,detailed,gemini-2.5-flash,20,67.530500,9.967000,86.814500,11734.600000,0.360604


In [15]:
# Cost-quality ratio (lower is better = more efficient)
ab_ratio = get_ab_test_cost_quality_ratio()
if not ab_ratio.empty:
    print("\n📈 Cost-Quality Ratio (lower = better value):")
    display(ab_ratio.sort_values('cost_per_quality_point'))
    
    fig = px.scatter(
        ab_ratio,
        x='avg_cost',
        y='avg_quality',
        color='prompt_variant',
        symbol='model',
        size='runs',
        title='Cost vs Quality by Prompt Variant',
        labels={'avg_cost': 'Average Cost ($)', 'avg_quality': 'Quality Score'},
        hover_data=['ab_test_name', 'runs']
    )
    fig.show()


📈 Cost-Quality Ratio (lower = better value):


,ab_test_name,prompt_variant,model,runs,avg_cost,avg_quality,cost_per_quality_point
0,generation_style,concise,gemini-2.5-flash,20,0.000061,90.503000,0.066610
1,generation_style,concise,gemini-2.5-pro,20,0.000675,93.999000,0.717276
2,generation_style,control,gemini-2.5-flash,20,0.000887,87.631500,1.012207
3,generation_style,cot,gemini-2.5-flash,20,0.001147,87.094000,1.319363
4,generation_style,detailed,gemini-2.5-flash,20,0.001539,86.814500,1.775368
5,generation_style,control,gemini-2.5-pro,9,0.008772,86.887778,10.100845
6,generation_style,cot,gemini-2.5-pro,13,0.009544,86.898462,10.983885
7,generation_style,detailed,gemini-2.5-pro,14,0.011832,87.077143,13.591049


In [16]:
# Statistical significance test between variants
if not runs_df.empty and 'prompt_variant' in runs_df.columns:
    ab_runs = runs_df[runs_df['ab_test_name'].notna()]
    if len(ab_runs) > 0:
        variants = ab_runs['prompt_variant'].unique()
        if len(variants) >= 2:
            print("\n📊 Statistical Significance Tests:")
            for i, v1 in enumerate(variants):
                for v2 in variants[i+1:]:
                    costs_v1 = ab_runs[ab_runs['prompt_variant'] == v1]['total_cost']
                    costs_v2 = ab_runs[ab_runs['prompt_variant'] == v2]['total_cost']
                    if len(costs_v1) > 5 and len(costs_v2) > 5:
                        t_stat, p_value = stats.ttest_ind(costs_v1, costs_v2)
                        sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else ''
                        print(f"  {v1} vs {v2}: p={p_value:.4f} {sig}")


📊 Statistical Significance Tests:
  control vs concise: p=0.0000 ***
  control vs detailed: p=0.0392 *
  control vs cot: p=0.2850 
  concise vs detailed: p=0.0000 ***
  concise vs cot: p=0.0000 ***
  detailed vs cot: p=0.2638 


## 8. Document Analysis & Vulnerability Detection

Analyze how well different pipelines detect security vulnerabilities.

In [17]:
# Ground truth vulnerability counts
print("🔒 Vulnerability Ground Truth Summary:")
print("="*50)

doc_ids = get_all_document_ids()
gt_summary = []

for doc_id in doc_ids:
    vulns = get_vulnerabilities(doc_id)
    severity_counts = count_by_severity(doc_id)
    gt_summary.append({
        'document': vulns.document_name[:40],
        'total_vulns': vulns.total_count,
        'critical': severity_counts.get('critical', 0),
        'high': severity_counts.get('high', 0),
        'medium': severity_counts.get('medium', 0),
        'low': severity_counts.get('low', 0),
    })

gt_df = pd.DataFrame(gt_summary)
display(gt_df)

print(f"\nTotal vulnerabilities across all documents: {gt_df['total_vulns'].sum()}")

🔒 Vulnerability Ground Truth Summary:


,document,total_vulns,critical,high,medium,low
0,User Authentication Module (user_auth.py,14,4,6,4,0
1,Flask REST API (api_server.py),18,6,6,3,3
2,Kubernetes Deployment (kubernetes-deploy,18,10,3,5,0
3,AWS Terraform (aws-infrastructure.tf),20,7,6,6,1
4,Bank Login Page (login-page.html),15,6,5,3,1
5,Application Config (app-config.json),20,8,8,4,0
6,Application Dockerfile,15,5,3,3,4
7,Docker Compose (docker-compose.yaml),17,7,9,1,0
8,Architecture Spec (architecture-spec.md),25,14,8,3,0



Total vulnerabilities across all documents: 162


In [18]:
# Severity distribution visualization
severity_totals = gt_df[['critical', 'high', 'medium', 'low']].sum()

fig = px.pie(
    values=severity_totals.values,
    names=severity_totals.index,
    title='Vulnerability Distribution by Severity',
    color=severity_totals.index,
    color_discrete_map={
        'critical': '#dc3545',
        'high': '#fd7e14',
        'medium': '#ffc107',
        'low': '#28a745'
    }
)
fig.show()

In [19]:
# Document workflow analysis
doc_runs = runs_df[runs_df['workflow'] == 'document']
if not doc_runs.empty:
    print("📄 Document Analysis Pipeline Comparison:")
    
    doc_summary = doc_runs.groupby(['pipeline', 'model']).agg({
        'total_cost': ['mean', 'std'],
        'total_latency_ms': 'mean',
        'num_stages': 'first',
        'id': 'count'
    }).round(4)
    doc_summary.columns = ['avg_cost', 'std_cost', 'avg_latency', 'stages', 'runs']
    doc_summary = doc_summary.reset_index()
    display(doc_summary)
    
    fig = px.bar(
        doc_summary,
        x='pipeline',
        y='avg_cost',
        color='model',
        barmode='group',
        error_y='std_cost',
        title='Document Analysis Pipeline Costs',
        labels={'avg_cost': 'Average Cost ($)', 'pipeline': 'Pipeline'}
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()
else:
    print("No document workflow data. Run: python -m src.experiment --workflow document --model flash")

📄 Document Analysis Pipeline Comparison:


,pipeline,model,avg_cost,std_cost,avg_latency,stages,runs
0,doc_analysis_hybrid,gemini-2.5-flash,0.0177,0.0020,95427.50,3,20
1,doc_analysis_hybrid,gemini-2.5-pro,0.0176,0.0018,95151.25,3,20
2,doc_analysis_iterative,gemini-2.5-flash,0.0143,0.0018,97717.70,3,20
3,doc_analysis_iterative,gemini-2.5-pro,0.0300,0.0029,123374.75,3,20
4,doc_analysis_simple,gemini-2.5-flash,0.0022,0.0004,37953.00,2,20
5,doc_analysis_simple,gemini-2.5-pro,0.0148,0.0023,63401.70,2,20
6,doc_analysis_thorough,gemini-2.5-flash,0.0167,0.0035,145486.80,4,20
7,doc_analysis_thorough,gemini-2.5-pro,0.0467,0.0050,176434.80,4,20


## 9. Cost-Quality Tradeoffs

In [20]:
# Merge runs with quality scores
if not quality_df.empty and not runs_df.empty:
    runs_quality = runs_df.merge(quality_df, left_on='id', right_on='run_id', how='inner')
    
    if not runs_quality.empty:
        print(f"Runs with quality scores: {len(runs_quality)}")
        
        # Cost vs Quality scatter
        fig = px.scatter(
            runs_quality,
            x='total_cost',
            y='combined_score',
            color='pipeline',
            symbol='model',
            title='Cost vs Quality Score',
            labels={'total_cost': 'Cost ($)', 'combined_score': 'Quality Score'}
            # trendline='ols'
        )
        fig.show()
        
        # Calculate cost efficiency (quality per dollar)
        runs_quality['quality_per_dollar'] = runs_quality['combined_score'] / (runs_quality['total_cost'] * 1000 + 0.001)
        
        efficiency = runs_quality.groupby(['pipeline', 'model']).agg({
            'quality_per_dollar': 'mean',
            'combined_score': 'mean',
            'total_cost': 'mean'
        }).round(2)
        efficiency = efficiency.sort_values('quality_per_dollar', ascending=False)
        print("\n💰 Cost Efficiency Ranking (quality per dollar, higher is better):")
        display(efficiency)
else:
    print("Quality data not available. Run experiments with --llm-eval flag.")

Runs with quality scores: 735



💰 Cost Efficiency Ranking (quality per dollar, higher is better):


quality_per_dollar  combined_score  \
pipeline               model                                                  
verbosity_concise      gemini-2.5-flash             2420.67           86.90   
context_short          gemini-2.5-flash              606.62           90.71   
ab_generation          gemini-2.5-flash              563.41           88.01   
verbosity_concise      gemini-2.5-pro                295.98           84.80   
react_research         gemini-2.5-flash              220.23           92.86   
                       gemini-2.5-pro                214.80           91.13   
context_short          gemini-2.5-pro                 81.66           90.22   
self_correcting        gemini-2.5-flash               66.21           85.12   
ab_generation          gemini-2.5-pro                 65.52           89.48   
self_correcting        gemini-2.5-pro                 65.50           84.85   
doc_analysis_simple    gemini-2.5-flash               39.81           85.82   
context_long           gemini-2.5-flash               39.31           78.82   
verbosity_cot          gemini-2.5-flash               24.86           87.47   
self_correcting_hybrid gemini-2.5-pro                 21.18           85.69   
                       gemini-2.5-flash               18.81           86.01   
react_hybrid           gemini-2.5-flash               13.12           87.45   
                       gemini-2.5-pro                 11.75           89.58   
multiturn_3            gemini-2.5-flash               10.86           46.38   
multiturn_5            gemini-2.5-flash                8.64           59.99   
hybrid_cot             gemini-2.5-flash                7.89           86.84   
                       gemini-2.5-pro                  7.78           87.32   
context_long           gemini-2.5-pro                  6.26           87.98   
doc_analysis_simple    gemini-2.5-pro                  6.11           88.13   
doc_analysis_iterative gemini-2.5-flash                6.06           85.45   
doc_analysis_thorough  gemini-2.5-flash                5.19           83.57   
doc_analysis_hybrid    gemini-2.5-pro                  4.74           82.64   
                       gemini-2.5-flash                4.65           82.75   
verbosity_cot          gemini-2.5-pro                  3.10           86.92   
doc_analysis_iterative gemini-2.5-pro                  2.94           87.44   
doc_analysis_thorough  gemini-2.5-pro                  1.81           83.58   
multiturn_3            gemini-2.5-pro                  1.44           49.50   
multiturn_5            gemini-2.5-pro                  1.15           75.71   

                                         total_cost  
pipeline               model                         
verbosity_concise      gemini-2.5-flash        0.00  
context_short          gemini-2.5-flash        0.00  
ab_generation          gemini-2.5-flash        0.00  
verbosity_concise      gemini-2.5-pro          0.00  
react_research         gemini-2.5-flash        0.00  
                       gemini-2.5-pro          0.00  
context_short          gemini-2.5-pro          0.00  
self_correcting        gemini-2.5-flash        0.00  
ab_generation          gemini-2.5-pro          0.01  
self_correcting        gemini-2.5-pro          0.00  
doc_analysis_simple    gemini-2.5-flash        0.00  
context_long           gemini-2.5-flash        0.00  
verbosity_cot          gemini-2.5-flash        0.00  
self_correcting_hybrid gemini-2.5-pro          0.00  
                       gemini-2.5-flash        0.01  
react_hybrid           gemini-2.5-flash        0.01  
                       gemini-2.5-pro          0.01  
multiturn_3            gemini-2.5-flash        0.00  
multiturn_5            gemini-2.5-flash        0.01  
hybrid_cot             gemini-2.5-flash        0.01  
                       gemini-2.5-pro          0.01  
context_long           gemini-2.5-pro          0.01  
doc_analysis_simple    gemini-2.5-pro          0.01  
doc_

## 10. RAG Pipeline Analysis

Analyze retrieval-augmented generation workflows with real FAISS embeddings.

## 10. Key Findings & Recommendations

In [21]:
print("\n" + "="*60)
print("📋 KEY FINDINGS SUMMARY")
print("="*60)

if not runs_df.empty:
    # Most expensive pipeline
    pipeline_costs = runs_df.groupby('pipeline')['total_cost'].mean().sort_values(ascending=False)
    print(f"\n💸 Most expensive pipeline: {pipeline_costs.index[0]} ({format_cost(pipeline_costs.iloc[0])}/run)")
    print(f"💵 Cheapest pipeline: {pipeline_costs.index[-1]} ({format_cost(pipeline_costs.iloc[-1])}/run)")
    
    # Model comparison
    model_avg = runs_df.groupby('model')['total_cost'].mean()
    if len(model_avg) >= 2:
        ratio = model_avg.max() / model_avg.min()
        print(f"\n🤖 Model cost ratio: {ratio:.1f}x ({model_avg.idxmax()} vs {model_avg.idxmin()})")
    
    # Total spent
    total = runs_df['total_cost'].sum()
    print(f"\n💰 Total experiment cost: {format_cost(total)}")
    print(f"📊 Total runs: {len(runs_df):,}")
    print(f"📈 Average cost per run: {format_cost(total/len(runs_df))}")

if not ab_summary.empty:
    print("\n🔬 A/B Test Insights:")
    best_variant = ab_summary.loc[ab_summary['avg_cost'].idxmin()]
    print(f"   Most cost-efficient variant: {best_variant['prompt_variant']}")

print("\n" + "="*60)


📋 KEY FINDINGS SUMMARY

💸 Most expensive pipeline: multiturn_5 ($0.0365/run)
💵 Cheapest pipeline: verbosity_concise ($0.000162/run)

🤖 Model cost ratio: 3.1x (gemini-2.5-pro vs gemini-2.5-flash)

💰 Total experiment cost: $8.07
📊 Total runs: 736
📈 Average cost per run: $0.0110

🔬 A/B Test Insights:
   Most cost-efficient variant: concise



In [22]:
# Export summary data
print("\n📁 Exporting summary data...")

# Save to CSV for external analysis
if not runs_df.empty:
    runs_df.to_csv('../data/runs_export.csv', index=False)
    print("   Saved: data/runs_export.csv")

if not stages_df.empty:
    stages_df.to_csv('../data/stages_export.csv', index=False)
    print("   Saved: data/stages_export.csv")

if not ab_summary.empty:
    ab_summary.to_csv('../data/ab_tests_export.csv', index=False)
    print("   Saved: data/ab_tests_export.csv")

print("\n✅ Analysis complete!")


📁 Exporting summary data...
   Saved: data/runs_export.csv
   Saved: data/stages_export.csv
   Saved: data/ab_tests_export.csv

✅ Analysis complete!
